# STAGE 1 Training - Kaggle T4
Target: mAP 80% → 82-86%

In [ ]:
# Clone repo
!git clone https://github.com/Khanhhh239/Model_XVLM_Training.git
!ls -la Model_XVLM_Training/

In [ ]:
import os
os.chdir('/kaggle/working/Model_XVLM_Training/trainv4')
print(f"Current dir: {os.getcwd()}")
!ls -la

In [ ]:
# Install
!pip install -q -r requirements.txt
!pip install -q albumentations
!pip install -q -e .

In [ ]:
# List datasets
import os
print("📦 Datasets:")
for name in os.listdir('/kaggle/input'):
    print(f"  - {name}")

In [ ]:
from pathlib import Path
import shutil

Path("data/checkpoints").mkdir(parents=True, exist_ok=True)

print("🔍 Finding files...\n")

# Find checkpoint
ckpt_datasets = [d for d in os.listdir('/kaggle/input') if 'ckpt' in d.lower()]
if ckpt_datasets:
    ckpt_dir = f"/kaggle/input/{ckpt_datasets[0]}"
    print(f"✓ Checkpoint dataset: {ckpt_datasets[0]}")
    !ls -lh {ckpt_dir}
    
    if Path(f"{ckpt_dir}/best.pth").exists():
        shutil.copy(f"{ckpt_dir}/best.pth", "data/checkpoints/best.pth")
        print("  ✓ Copied best.pth")

# Find bbox
bbox_datasets = [d for d in os.listdir('/kaggle/input') if 'bbox' in d.lower()]
if bbox_datasets:
    bbox_dir = f"/kaggle/input/{bbox_datasets[0]}"
    print(f"\n✓ Bbox dataset: {bbox_datasets[0]}")
    !ls -lh {bbox_dir}
    
    bbox_files = [f for f in os.listdir(bbox_dir) if 'boxes' in f.lower()]
    if bbox_files:
        shutil.copy(f"{bbox_dir}/{bbox_files[0]}", f"data/{bbox_files[0]}")
        print(f"  ✓ Copied {bbox_files[0]}")

# Find data
data_datasets = [d for d in os.listdir('/kaggle/input') if 'aicity' in d.lower() or '30k' in d.lower()]
if data_datasets:
    data_dir = f"/kaggle/input/{data_datasets[0]}"
    print(f"\n✓ Data dataset: {data_datasets[0]}")
    !ls -lh {data_dir}
    
    # Extract tar.zst
    tar_files = [f for f in os.listdir(data_dir) if '.tar' in f]
    if tar_files:
        print(f"\n📦 Extracting {tar_files[0]}...")
        !tar -I zstd -xf {data_dir}/{tar_files[0]} -C data/ || tar -xf {data_dir}/{tar_files[0]} -C data/
        print("✓ Extracted!")

In [ ]:
# Check structure
print("📁 Data structure:")
!ls -lh data/

In [ ]:
# Find paths
import glob

print("🔍 Finding paths...\n")

manifest = glob.glob("data/**/*train*hard*.jsonl", recursive=True) + glob.glob("data/**/*train*hard*.parquet", recursive=True)
manifest = manifest[0] if manifest else None
print(f"Manifest: {manifest}")

vitpose = glob.glob("data/**/*vitpose*.json", recursive=True)
vitpose = vitpose[0] if vitpose else None
print(f"VitPose: {vitpose}")

images = glob.glob("data/**/images", recursive=True) + glob.glob("data/**/train_webp", recursive=True)
images = images[0] if images else None
print(f"Images: {images}")

boxes = glob.glob("data/**/boxes*.json*", recursive=True)
boxes = boxes[0] if boxes else None
print(f"Boxes: {boxes}")

In [ ]:
# Create config
import yaml

with open("configs/stage1_30k_kaggle_t4.yaml", 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data']['manifest'] = manifest
cfg['data']['image_root'] = images + '/' if images else 'data/images/'
cfg['data']['vitpose_json'] = vitpose if vitpose else 'data/vitpose.json'
cfg['data']['boxes_json'] = boxes if boxes else 'data/boxes.json'
cfg['model']['checkpoint'] = 'data/checkpoints/best.pth'
cfg['train']['batch_size'] = 16

with open('configs/runtime.yaml', 'w') as f:
    yaml.dump(cfg, f)

print("✓ Config created!")

In [ ]:
# Verify
print("🔍 Verifying...\n")
for name, path in [
    ('Manifest', manifest),
    ('VitPose', vitpose),
    ('Boxes', boxes),
    ('Images', images),
    ('Checkpoint', 'data/checkpoints/best.pth'),
]:
    exists = path and Path(path).exists()
    print(f"{'✓' if exists else '❌'} {name}: {path}")
    if not exists:
        raise FileNotFoundError(f"{name} not found!")

print("\n✅ All verified!")

In [ ]:
# Sanity check
print("🧪 Sanity check...\n")
!python scripts/train.py --config configs/runtime.yaml --init-from data/checkpoints/best.pth --overfit-one-batch
print("\n✅ Passed!")

In [ ]:
# TRAINING
print("🚀 Training...\n")
!python scripts/train.py --config configs/runtime.yaml --init-from data/checkpoints/best.pth --max-hours 11.5
print("\n🎉 Done!")

In [ ]:
# Results
import torch
ckpt = torch.load("outputs/stage1_30k_t4/best.pth", map_location='cpu')
print("📊 Results:")
print(f"  mAP: {ckpt['report']['mAP']*100:.2f}%")
shutil.copy("outputs/stage1_30k_t4/best.pth", "/kaggle/working/stage1_best.pth")
print("\n✓ Saved to /kaggle/working/stage1_best.pth")